# Текущий прогресс разметки
Нажмите **Run → Run All Cells**, чтобы загрузить свежий снимок из S3. Airflow обновляет его примерно раз в 5 минут. Здесь только чтение: запросы Gemini не запускаются.

**Ответ запроса**, **уникальное аудио** и **кандидат 3/3** считаются отдельно. Совпавшие ответы — кандидаты на проверку, а не human-gold. Проход с исходной меткой зависит от этой метки.


In [ ]:
import json, hashlib
import pandas as pd
from IPython.display import Audio, display
from lalm.annotation.inspection import open_snapshot
from lalm.annotation.contract import TRIAD_VERSION, triad_decision
if 'db' in globals():
    db.close()
    cache.cleanup()
s3, snapshot, cache, db = open_snapshot()
print('Снимок:', snapshot['snapshot_at'])
age = pd.Timestamp.now(tz='UTC') - pd.Timestamp(snapshot['snapshot_at'])
print('Возраст снимка:', round(age.total_seconds()/60, 1), 'мин')
if age.total_seconds() > 900:
    print('ВНИМАНИЕ: снимок старше 15 минут — проверьте Airflow.')
passes = db.sql('SELECT * FROM passes').df()
mask = passes.batch_id.fillna('').str.startswith(TRIAD_VERSION+'-')
current = passes[mask].copy()
examples = db.sql('SELECT * FROM examples').df()
examples = examples[examples.batch_id.fillna('').str.startswith(TRIAD_VERSION+'-')].copy()
old_texts = passes[~mask & passes.gemini_text.notna()].groupby('sample_id').gemini_text.agg(list).to_dict()
records = []
for (batch, sample), group in current.groupby(['batch_id','sample_id']):
    results = {}
    for r in group.itertuples():
        if isinstance(r.gemini_text, str):
            spans = json.loads(r.uncertain_spans) if isinstance(r.uncertain_spans, str) else []
            results[r.prompt_variant] = {'verbatim_text': r.gemini_text, 'language': r.language or '', 'uncertain_spans': spans}
    decision = triad_decision(results, old_texts.get(sample, []))
    if decision == 'waiting_remaining' and group.original_annotation_status.eq('not_linked_or_not_available').all():
        decision = 'missing_source_annotation'
    if (group.processing_state.isin(['NEEDS_REVIEW','PERMANENT_FAILED']) & group.gemini_text.isna()).any():
        decision = 'rejected_failed_pass'
    records.append({'batch_id':batch, 'sample_id':sample, 'decision':decision})
status = pd.DataFrame(records, columns=['batch_id','sample_id','decision'])
examples = examples.merge(status, on=['batch_id','sample_id'], how='left', validate='one_to_one')
labels = {'candidate_3of3':'Совпали 3/3: кандидаты', 'waiting_first':'Ожидают первый ответ',
          'waiting_remaining':'Ожидают остальные проходы', 'missing_source_annotation':'Нет исходной метки',
          'rejected_first':'Первый ответ: не ru / неопределённость / пусто',
          'rejected_quality':'Не прошли проверку языка / определённости',
          'rejected_disagreement':'Расхождение трёх ответов', 'rejected_prior_conflict':'Конфликт с прежними ответами',
          'rejected_failed_pass':'Ошибка или отклонённый ответ'}
examples['статус'] = examples.decision.map(labels).fillna(examples.decision)


## Сколько обработано
Сводка относится к текущему протоколу precision-ru-triad-v1. Полного числа будущих фрагментов ещё нет: архивы продолжают разбираться, поэтому общий процент готовности не вычисляется.

In [ ]:
summary = {
    'Уникальных аудио в очереди': current.audio_sha256.nunique(),
    'Всего заданий-проходов': len(current),
    'Сохранённых текстовых ответов': int(current.gemini_text.notna().sum()),
    'В работе (на момент снимка)': int(current.processing_state.eq('RUNNING').sum()),
    'Ожидают / повтор после ошибки': int(current.processing_state.isin(['PENDING','RETRYABLE_FAILED']).sum()),
    'Уникальных аудио — кандидаты 3/3': examples.loc[examples.decision.eq('candidate_3of3'),'audio_sha256'].nunique(),
    'Подтверждено человеком': int(examples.human_verified.sum()),
}
display(pd.Series(summary, name='Количество').to_frame())
display(examples.groupby('статус', dropna=False).size().rename('Примеров').to_frame())
display(pd.crosstab(examples.corpus, examples['статус']))


## Часы аудио
Длительность берётся из сохранённых ответов и считается **один раз на SHA256 аудио**, независимо от числа проходов. У ожидающих аудио без ответа длительность в снимке пока неизвестна: они показаны отдельно, а не приняты за нулевую длительность.

Это часы подготовленных фрагментов: перекрытия соседних сегментов не вычитаются. «Кандидаты 3/3» ещё не означают проверенную человеком разметку.


In [ ]:
durations = current.groupby('audio_sha256', dropna=True).audio_seconds.max()
audio_rows = examples.sort_values('updated_at').drop_duplicates('audio_sha256', keep='last').copy()
audio_rows['audio_seconds'] = audio_rows.audio_sha256.map(durations)
with_text = set(current.loc[current.gemini_text.notna(), 'audio_sha256'])
audio_rows['has_answer'] = audio_rows.audio_sha256.isin(with_text)
audio_rows['candidate'] = audio_rows.decision.eq('candidate_3of3')
hours = pd.Series({
    'Известная длительность уникальных фрагментов, ч': audio_rows.audio_seconds.sum()/3600,
    'Есть хотя бы один сохранённый ответ, ч': audio_rows.loc[audio_rows.has_answer, 'audio_seconds'].sum()/3600,
    'Кандидаты 3/3, ч': audio_rows.loc[audio_rows.candidate, 'audio_seconds'].sum()/3600,
    'Подтверждено человеком, ч': audio_rows.loc[audio_rows.human_verified, 'audio_seconds'].sum()/3600,
}, name='Часы')
display(hours.round(3).to_frame())
print('Аудио с неизвестной длительностью:', int(audio_rows.audio_seconds.isna().sum()))
by_corpus = audio_rows.assign(
    known_hours=audio_rows.audio_seconds/3600,
    answered_hours=audio_rows.audio_seconds.where(audio_rows.has_answer, 0)/3600,
    candidate_hours=audio_rows.audio_seconds.where(audio_rows.candidate, 0)/3600,
    unknown_duration=audio_rows.audio_seconds.isna(),
).groupby('corpus', dropna=False).agg(
    unique_audio=('audio_sha256','size'),
    known_hours=('known_hours','sum'),
    answered_hours=('answered_hours','sum'),
    candidate_3of3_hours=('candidate_hours','sum'),
    unknown_duration=('unknown_duration','sum'),
)
display(by_corpus.round(3))
display(audio_rows.groupby('статус', dropna=False).agg(
    unique_audio=('audio_sha256','size'), seconds=('audio_seconds','sum')
).assign(hours=lambda x:x.seconds/3600).drop(columns='seconds').round(3))


## Аккаунты, ошибки и недавние ответы
Это активность из снимка, а не остаток квоты. NEEDS_REVIEW с pilot_review_only — нормальный сохранённый результат, ожидающий проверки. Число ответов за интервал считается по updated_at, а не по полной истории событий.

In [ ]:
display(current.groupby(['account_id','processing_state'], dropna=False).agg(
    requests=('job_id','size'), texts=('gemini_text','count'), last_update=('updated_at','max')))
errors = current[current.error_code.notna() & current.error_code.ne('pilot_review_only')]
display(errors.groupby(['account_id','error_code'], dropna=False).size().rename('Запросов').to_frame())
answered = current[current.gemini_text.notna()].copy()
answered['интервал UTC'] = pd.to_datetime(answered.updated_at, utc=True).dt.floor('30min')
display(answered.groupby('интервал UTC').size().rename('Ответов').tail(24).to_frame())


## Входные данные и публикации
Исходный объект может быть архивом с множеством аудио. Его нельзя считать одним готовым примером. Пустые публикации означают, что обучающий релиз ещё не выпущен.

In [ ]:
display(db.sql("SELECT corpus, preparation_state, count(*) AS source_objects, sum(samples_prepared) AS prepared_samples, sum(preparation_errors) AS preparation_errors FROM sources GROUP BY ALL ORDER BY corpus, preparation_state").df())
display(db.sql('SELECT * FROM publications ORDER BY created_at DESC').df())


## Выбрать примеры
По умолчанию показаны последние кандидаты, где совпали все три ответа. Измените фильтры и запустите эту ячейку и следующую. Для всех статусов поставьте decision_filter = None; для расхождений — rejected_disagreement.

In [ ]:
decision_filter = 'candidate_3of3'
corpus_filter = None
sample_search = ''  # Полный ID или его начало.
limit = 30
selected = examples.copy()
if decision_filter:
    selected = selected[selected.decision.eq(decision_filter)]
if corpus_filter:
    selected = selected[selected.corpus.eq(corpus_filter)]
if sample_search:
    selected = selected[selected.sample_id.str.startswith(sample_search, na=False)]
print('Найдено:', len(selected))
selected = selected.sort_values('updated_at', ascending=False).head(limit).reset_index(drop=True)
columns = ['sample_id','corpus','статус','original_annotation_json','gemini_verbatim',
           'gemini_coverage','gemini_source_correction','passes_with_text','unfinished_passes','error_codes']
with pd.option_context('display.max_colwidth', 180):
    display(selected[columns])


## Прослушать и сравнить
Выберите номер строки из таблицы выше. Плеер загружает только выбранный фрагмент и проверяет SHA256. Все три текста показываются полностью.

In [ ]:
example_index = 0
if selected.empty:
    print('Для этих фильтров примеров пока нет.')
else:
    row = selected.iloc[example_index]
    print('Пример:', row.sample_id, '|', row['статус'])
    print('Исходная метка:', row.original_annotation_json)
    group = current[(current.batch_id == row.batch_id) & (current.sample_id == row.sample_id)]
    for item in group.sort_values('prompt_variant').itertuples():
        print('\n', item.prompt_variant, '| язык:', item.language, '|', item.processing_state, '|', item.error_code)
        print(item.gemini_text if isinstance(item.gemini_text, str) else 'Нет принятого текста')
    response = s3.get_object(Bucket=row.audio_bucket, Key=row.audio_key)
    try:
        if response['ContentLength'] > 100 * 1024**2:
            raise ValueError('Фрагмент слишком большой для плеера')
        audio = response['Body'].read()
    finally:
        response['Body'].close()
    assert hashlib.sha256(audio).hexdigest() == row.audio_sha256
    display(Audio(audio))
    print('Источник и границы фрагмента:')
    display(json.loads(group.iloc[0].source_provenance or '{}'))
